In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from urllib.parse import quote
import re
import time
import random
from tqdm.auto import tqdm
import pandas as pd

In [85]:
def get_page_html(url):
    driver.get(url)
    time.sleep(random.uniform(1, 3))
    return driver.page_source

In [118]:
def parse_one_article(block):
    url_p = block['url']
    page_html = get_page_html(url_p)
    soup = BeautifulSoup(page_html, 'html.parser')

    descr = soup.find('table', {'class': 'card-descr-table'})
    if not descr:
        return block
    
    author_td = descr.find('td', {'itemprop': 'author'})
    block['author'] = author_td.text.strip() if author_td else 'Не указан'
    
    title_td = descr.find('td', {'itemprop': 'name'})
    block['title'] = title_td.text.strip() if title_td else 'Не указано'

    output_data = descr.find('th', string='Выходные данные')
    if output_data:
        output_data_value = output_data.find_next('td').text
        match = re.search(r'\d{4}', output_data_value)
        block['date'] = match.group(0) if match else 'Не указано'
    else:
        block['date'] = 'Не указано'
        
    return block

In [117]:
def parse_page_block(one_block):
    block = {}
    href = one_block.find('a').attrs['href']
    block['url'] = 'https://search.rsl.ru' + href
    return block

In [88]:
def get_total_pages(theme_encoded):
    url = f'https://search.rsl.ru/ru/search#t={theme_encoded}&p=1'
    page_html = get_page_html(url)
    soup = BeautifulSoup(page_html, 'html.parser')
    
    page_info = soup.find('div', {'class': 'rsl-search-info'})
    if page_info:
        text = page_info.get_text()
        match = re.search(r'страница\s+\d+\s*/\s*(\d+)', text)
        if match:
            return int(match.group(1))
    
    return 1

In [119]:
def get_nth_page(page_number, theme_encoded):
    url = f'https://search.rsl.ru/ru/search#t={theme_encoded}&p={page_number}'
    page_html = get_page_html(url)
    soup = BeautifulSoup(page_html, 'html.parser')

    posts_preview = soup.find_all('div', {'class': 'rsl-itemaction-link rsl-itemaction-description-link'})

    blocks = []
    for p_p in posts_preview:
        try:
            blocks.append(parse_page_block(p_p))
        except Exception as e:
            print(e)

    result = []
    for b in blocks:
        try:
            res = parse_one_article(b)
            result.append(res)
        except Exception as e:
            print(e)

    return result

In [120]:
def run_all(theme_encoded):
    total_pages = get_total_pages(theme_encoded)
    blocks = []
    
    for i in tqdm(range(total_pages)):
        blocks.extend(get_nth_page(i+1, theme_encoded))
    
    return blocks

In [110]:
options = Options()
options.add_argument("--headless")
driver = webdriver.Chrome(options=options)

In [111]:
themes = [
    'Ш|Ш1|Ш12/17|Ш160|Ш160.3',   # Чечено-дагестанская группа языков 
    'Ш|Ш1|Ш12/17|Ш160|Ш160.4'  # Лезгинские языки
]

In [112]:
for theme in themes:
    theme_encoded = quote(theme).replace('/', '%2F')
    blocks = run_all(theme_encoded)
    all_blocks.extend(blocks)

  0%|          | 0/2 [00:00<?, ?it/s]

https://search.rsl.ru/ru/search#t=%D0%A8%7C%D0%A81%7C%D0%A812%2F17%7C%D0%A8160%7C%D0%A8160.3&p=1
10
https://search.rsl.ru/ru/search#t=%D0%A8%7C%D0%A81%7C%D0%A812%2F17%7C%D0%A8160%7C%D0%A8160.3&p=2
20


  0%|          | 0/2 [00:00<?, ?it/s]

https://search.rsl.ru/ru/search#t=%D0%A8%7C%D0%A81%7C%D0%A812%2F17%7C%D0%A8160%7C%D0%A8160.4&p=1
10
https://search.rsl.ru/ru/search#t=%D0%A8%7C%D0%A81%7C%D0%A812%2F17%7C%D0%A8160%7C%D0%A8160.4&p=2
20


In [94]:
df = pd.DataFrame(all_blocks)
df.to_csv('publications_rsl.csv', index=False)

In [116]:
driver.quit()